# TrafficFlow-AI: Model Training
Explore the data, train the traffic volume model, and evaluate it.

In [ ]:
import sys; sys.path.append('..')
import joblib, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from src.preprocessing import load_data, build_preprocessor, FEATURES, TARGET
%matplotlib inline

## 1. Load data

In [ ]:
df = load_data()
print(df.shape)
df.head()

## 2. Explore

In [ ]:
df.groupby(['hour','junction'])['vehicles'].mean().unstack().plot(figsize=(10,4), title='Average traffic by hour')
plt.ylabel('vehicles / hour'); plt.show()

In [ ]:
df.groupby('weather')['vehicles'].mean().plot.bar(title='Average traffic by weather'); plt.show()

## 3. Train

In [ ]:
X, y = df[FEATURES], df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = Pipeline([('prep', build_preprocessor()), ('reg', HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=42))])
model.fit(X_tr, y_tr)

## 4. Evaluate

In [ ]:
pred = model.predict(X_te)
print('MAE:', round(mean_absolute_error(y_te, pred), 2), '| R2:', round(r2_score(y_te, pred), 4))
plt.figure(figsize=(5,5)); plt.scatter(y_te, pred, s=3, alpha=.4); plt.plot([0,y.max()],[0,y.max()],'r--')
plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.show()

## 5. Save

In [ ]:
joblib.dump(model, '../models/traffic_model.pkl')